In [1]:
import torch, torch_geometric
print(torch.__version__, torch_geometric.__version__)

C:\Users\Pranav\fraud-detection-platform\venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


2.14.1+cpu 2.8.1


In [2]:
import pandas as pd, numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

f = pd.read_parquet("data/features.parquet")
extra = (["TransactionID", "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4",
          "card5", "card6", "addr1", "addr2", "dist1", "P_emaildomain", "R_emaildomain"]
         + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)]
         + [f"M{i}" for i in range(1, 10)])
raw = pd.read_csv("data/train_transaction.csv", usecols=extra)
idt = pd.read_csv("data/train_identity.csv", usecols=["TransactionID", "DeviceInfo"])

df = f.merge(raw, on="TransactionID", how="left").merge(idt, on="TransactionID", how="left")
df = df.sort_values("event_timestamp").reset_index(drop=True)

cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"] + [f"M{i}" for i in range(1, 10)]
for c in cat_cols:
    df[c] = df[c].astype("category")

my_features = (["hour", "day_of_week", "tx_count_before", "secs_since_last", "amt_vs_avg",
                "device_changed", "addr_changed", "email_changed", "TransactionAmt",
                "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
               + cat_cols + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])

n = len(df)
a, b = int(n * 0.72), int(n * 0.80)   # same split as XGBoost
y = df["isFraud"].values
print(n, "transactions |", len(my_features), "features")

590540 transactions | 59 features


In [3]:
import torch

def chain_edges(key):
    """Arrow from the previous transaction with the same key to the current one."""
    t = pd.DataFrame({"k": key.values, "pos": np.arange(len(key))}).dropna(subset=["k"])
    t["prev"] = t.groupby("k")["pos"].shift(1)
    t = t.dropna(subset=["prev"])
    return t["prev"].astype("int64").values, t["pos"].values

combo = (df["P_emaildomain"].astype(object).astype(str) + "_" + df["addr1"].astype(str)).where(
    df["P_emaildomain"].notna() & df["addr1"].notna())

src, dst = [], []
for name, key in [("card", df["card_key"]), ("device", df["DeviceInfo"]), ("email+addr", combo)]:
    s, d = chain_edges(key)
    print(name, "edges:", len(s))
    src.append(s); dst.append(d)

edge_index = torch.tensor(np.vstack([np.concatenate(src), np.concatenate(dst)]), dtype=torch.long)

# node features: turn categories into numbers, compress big values, standardize
X = df[my_features].copy()
for c in cat_cols:
    X[c] = X[c].cat.codes.astype("float32").replace(-1, np.nan)
X = X.astype("float32").values
X = np.sign(X) * np.log1p(np.abs(X))          # signed log: tames huge values
mu, sd = np.nanmean(X[:a], axis=0), np.nanstd(X[:a], axis=0) + 1e-6   # statistics from TRAIN only
X = np.clip((X - mu) / sd, -5, 5)
X = np.nan_to_num(X, nan=0.0).astype("float32")

x = torch.tensor(X)
yt = torch.tensor(y, dtype=torch.float32)
print("graph:", x.shape, edge_index.shape)

card edges: 575695
device edges: 116880
email+addr edges: 430649
graph: torch.Size([590540, 59]) torch.Size([2, 1123224])


In [4]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GNN(torch.nn.Module):
    def __init__(self, d, h=64):
        super().__init__()
        self.c1, self.c2 = SAGEConv(d, h), SAGEConv(h, h)
        self.out = torch.nn.Linear(h, 1)
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.c1(x, ei)), 0.3, self.training)
        x = F.dropout(F.relu(self.c2(x, ei)), 0.3, self.training)
        return self.out(x).squeeze(-1)

torch.manual_seed(42)
model_g = GNN(x.shape[1])
opt = torch.optim.Adam(model_g.parameters(), lr=0.005, weight_decay=1e-5)
ratio = (y[:a] == 0).sum() / (y[:a] == 1).sum()
pos_w = torch.tensor(float(ratio) ** 0.5)       # softer fraud weight than XGBoost's 27
train_idx = torch.arange(0, a)

best_val, best_prob, bad = 0, None, 0
for epoch in range(1, 31):
    model_g.train(); opt.zero_grad()
    out = model_g(x, edge_index)
    loss = F.binary_cross_entropy_with_logits(out[train_idx], yt[train_idx], pos_weight=pos_w)
    loss.backward(); opt.step()

    model_g.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model_g(x, edge_index)).numpy()
    val_pr = average_precision_score(y[a:b], prob[a:b])
    print(f"epoch {epoch:2d}  loss {loss.item():.4f}  val PR-AUC {val_pr:.4f}")

    if val_pr > best_val:
        best_val, best_prob, bad = val_pr, prob.copy(), 0
        torch.save(model_g.state_dict(), "models/gnn.pt")
    else:
        bad += 1
        if bad >= 8: break      # early stopping

print("GNN alone  val:", round(best_val, 4),
      " test PR-AUC:", round(average_precision_score(y[b:], best_prob[b:]), 4))

epoch  1  loss 0.7527  val PR-AUC 0.0550
epoch  2  loss 0.6288  val PR-AUC 0.0754
epoch  3  loss 0.5464  val PR-AUC 0.1024
epoch  4  loss 0.4969  val PR-AUC 0.1308
epoch  5  loss 0.4770  val PR-AUC 0.1537
epoch  6  loss 0.4747  val PR-AUC 0.1630
epoch  7  loss 0.4769  val PR-AUC 0.1708
epoch  8  loss 0.4731  val PR-AUC 0.1812
epoch  9  loss 0.4631  val PR-AUC 0.1956
epoch 10  loss 0.4502  val PR-AUC 0.2152
epoch 11  loss 0.4375  val PR-AUC 0.2355
epoch 12  loss 0.4279  val PR-AUC 0.2504
epoch 13  loss 0.4219  val PR-AUC 0.2597
epoch 14  loss 0.4201  val PR-AUC 0.2666
epoch 15  loss 0.4196  val PR-AUC 0.2723
epoch 16  loss 0.4191  val PR-AUC 0.2776
epoch 17  loss 0.4175  val PR-AUC 0.2838
epoch 18  loss 0.4135  val PR-AUC 0.2901
epoch 19  loss 0.4095  val PR-AUC 0.2974
epoch 20  loss 0.4035  val PR-AUC 0.3038
epoch 21  loss 0.3994  val PR-AUC 0.3086
epoch 22  loss 0.3942  val PR-AUC 0.3117
epoch 23  loss 0.3922  val PR-AUC 0.3141
epoch 24  loss 0.3894  val PR-AUC 0.3165
epoch 25  loss 0

In [ ]:
import mlflow
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("models/xgb_v2.json")
px = xgb_model.predict_proba(df[my_features])[:, 1]

def rank(v): return pd.Series(v).rank(pct=True).values

best_alpha, best_score = 0, 0
for alpha in np.arange(0, 1.01, 0.1):                 # choose the blend on VALIDATION only
    s = alpha * rank(best_prob[a:b]) + (1 - alpha) * rank(px[a:b])
    pr = average_precision_score(y[a:b], s)
    if pr > best_score: best_alpha, best_score = alpha, pr

blend_test = best_alpha * rank(best_prob[b:]) + (1 - best_alpha) * rank(px[b:])
res = {
    "xgb_test_pr": average_precision_score(y[b:], px[b:]),
    "gnn_test_pr": average_precision_score(y[b:], best_prob[b:]),
    "blend_test_pr": average_precision_score(y[b:], blend_test),
    "blend_test_roc": roc_auc_score(y[b:], blend_test),
}
print("best GNN weight (alpha):", round(best_alpha, 1))
for k, v in res.items(): print(k, round(v, 4))

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")
with mlflow.start_run(run_name="gnn_xgb_blend"):
    mlflow.log_param("alpha", best_alpha)
    mlflow.log_metrics(res)